# Notebook 03 — Pipeline de modelado y evaluación (CF) a corto plazo

Este notebook implementa un pipeline reproducible para **entrenar, validar y comparar** modelos de predicción a corto plazo (48–72h) sobre series temporales de **huella de carbono (CF)** asociada al consumo eléctrico.

**Modelos a evaluar (en orden):**
1. Baselines (Naive)
2. ARIMA (no estacional)
3. SARIMA (estacional)
4. SARIMAX (con variables exógenas, si están disponibles)

**Decisiones clave del diseño experimental:**
- La partición es **temporal** para evitar fuga de información.
- La evaluación es **walk-forward / rolling origin** para simular el uso real (reforecast periódico).
- Se evalúan horizontes multi-step de **48h y 72h** con resolución de **15 minutos**.
- Métricas principales: **MAE** y **RMSE**.

In [1]:

import numpy as np
import pandas as pd

from dataclasses import dataclass
from typing import Dict, List, Tuple, Optional

from sklearn.metrics import mean_absolute_error, mean_squared_error

# Statsmodels para ARIMA
from statsmodels.tsa.arima.model import ARIMA

## Carga de datos

Los datos han sido previamente procesados y divididos en conjuntos temporales:

- Train: 2022–2023
- Validación: 2024
- Test: 2025

Esta separación temporal evita fuga de información y permite evaluar los modelos en
condiciones similares a un escenario real de predicción futura.

In [2]:
from pathlib import Path
import pandas as pd

DATA_DIR = Path("data_processed")

train_path = DATA_DIR / "train_2022_2023.parquet"
val_path   = DATA_DIR / "val_2024.parquet"
test_path  = DATA_DIR / "test_2025.parquet"


train = pd.read_parquet(DATA_DIR / "train_2022_2023.parquet").sort_index()
val   = pd.read_parquet(DATA_DIR / "val_2024.parquet").sort_index()
test  = pd.read_parquet(DATA_DIR / "test_2025.parquet").sort_index()

print("Train:", train.shape, "|", train.index.min(), "->", train.index.max())
print("Val:  ", val.shape,   "|", val.index.min(),   "->", val.index.max())
print("Test: ", test.shape,  "|", test.index.min(),  "->", test.index.max())

train.head()

Train: (70080, 1) | 2022-01-01 00:00:00+00:00 -> 2023-12-31 23:45:00+00:00
Val:   (35136, 1) | 2024-01-01 00:00:00+00:00 -> 2024-12-31 23:45:00+00:00
Test:  (35040, 1) | 2025-01-01 00:00:00+00:00 -> 2025-12-31 23:45:00+00:00


,y
timestamp,
2022-01-01 00:00:00+00:00,120.79
2022-01-01 00:15:00+00:00,120.79
2022-01-01 00:30:00+00:00,120.79
2022-01-01 00:45:00+00:00,120.80
2022-01-01 01:00:00+00:00,120.04


## Variable objetivo

La variable objetivo es la huella de carbono operacional (`y`), expresada en gCO₂eq/kWh.

Se trabaja únicamente con la serie temporal univariante en esta fase para:

1. Establecer modelos base de referencia.
2. Evaluar la capacidad predictiva intrínseca de la serie.
3. Facilitar comparaciones posteriores con modelos estacionales y deep learning.

In [3]:
TARGET_COL = "y"

y_train = train[TARGET_COL].astype(float)
y_val   = val[TARGET_COL].astype(float)
y_test  = test[TARGET_COL].astype(float)

# Asegurar índice temporal
y_train.index = pd.to_datetime(y_train.index)
y_val.index   = pd.to_datetime(y_val.index)
y_test.index  = pd.to_datetime(y_test.index)

print(y_train.index.min(), "->", y_train.index.max())

2022-01-01 00:00:00+00:00 -> 2023-12-31 23:45:00+00:00


In [4]:
from dataclasses import dataclass

@dataclass
class WFConfig:
    step: int
    min_history: int
    max_fits: int

## Configuración temporal del problema

La serie tiene resolución de **15 minutos**, por lo que:

- 1 hora = 4 pasos
- 24 horas = 96 pasos
- 48 horas = 192 pasos
- 72 horas = 288 pasos

El periodo estacional diario es de 96 pasos, lo que será relevante para modelos
estacionales posteriores (SARIMA).

In [5]:
import numpy as np

FREQ_MIN = 15
STEPS_PER_HOUR = 60 // FREQ_MIN
SEASONAL_PERIOD = 24 * STEPS_PER_HOUR  # 96

HORIZONS = {
    "48h": 48 * STEPS_PER_HOUR,
    "72h": 72 * STEPS_PER_HOUR
}

SEASONAL_PERIOD, HORIZONS

(96, {'48h': 192, '72h': 288})

In [6]:
cfg_final = WFConfig(
    step=SEASONAL_PERIOD,            # reforecast diario
    min_history=14 * SEASONAL_PERIOD,# mínimo 14 días dentro de 2024 antes de empezar
    max_fits=120                     # nº de reentrenos (ajusta si quieres)
)

cfg_final

WFConfig(step=96, min_history=1344, max_fits=120)

## Métricas de evaluación

Se utilizan:

- MAE (Mean Absolute Error)
- RMSE (Root Mean Squared Error)

RMSE penaliza más los errores grandes, mientras que MAE proporciona una medida
más interpretable del error medio absoluto.

In [7]:
from sklearn.metrics import mean_absolute_error, mean_squared_error

def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

def compute_metrics(y_true, y_pred):
    return {
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": rmse(y_true, y_pred)
    }

## Estrategia de evaluación: walk-forward

Se utiliza un esquema walk-forward con reentrenamiento periódico para simular un
escenario real de predicción operativa.

Cada día (96 pasos):

1. Se entrena el modelo con el histórico disponible.
2. Se generan predicciones para 48h y 72h.
3. Se calculan métricas.

Este enfoque proporciona una evaluación más robusta que una única partición.

# Modelo ARIMA

ARIMA permite modelar dependencias temporales mediante componentes:

- AR (autoregresivo)
- I (diferenciación)
- MA (media móvil)

Se realiza una búsqueda de hiperparámetros moderada para equilibrar rendimiento y
coste computacional.



### Selección automática del orden ARIMA

El orden del modelo ARIMA se determinó mediante el algoritmo `auto_arima`, basado en
criterios de información (AIC). Para reducir el coste computacional, la estimación se
realizó sobre una ventana temporal representativa del conjunto de entrenamiento, dado
que la identificación de parámetros depende principalmente de la estructura de
autocorrelación local.

Se permitió una exploración de valores moderados de p y q, priorizando configuraciones
capaces de capturar la dinámica temporal sin introducir una complejidad excesiva que
pudiera provocar sobreajuste o inestabilidad numérica.

El modelo final se entrena posteriormente utilizando todo el histórico disponible
en cada instante durante la evaluación walk-forward.

In [8]:
import pmdarima as pm

# Usamos solo una ventana del train para acelerar
y_sample = y_train.loc[y_train.index >= y_train.index.max() - pd.Timedelta(days=30)]   # últimos 30 días

auto_model = pm.auto_arima(
    y_sample.values,
    
    seasonal=False,          # ARIMA no estacional
    start_p=0,
    start_q=0,
    max_p=6,
    max_q=6,
    
    d=None,                  # estimar automáticamente
    max_d=2,
    
    stepwise=True,          
    information_criterion="aic",
    
    suppress_warnings=True,
    error_action="ignore",
    trace=True               
)

print(auto_model.summary())

Performing stepwise search to minimize aic
 ARIMA(0,1,0)(0,0,0)[0] intercept   : AIC=14367.843, Time=0.04 sec
 ARIMA(1,1,0)(0,0,0)[0] intercept   : AIC=13325.760, Time=0.09 sec
 ARIMA(0,1,1)(0,0,0)[0] intercept   : AIC=13509.429, Time=0.17 sec
 ARIMA(0,1,0)(0,0,0)[0]             : AIC=14365.873, Time=0.03 sec
 ARIMA(2,1,0)(0,0,0)[0] intercept   : AIC=13327.737, Time=0.15 sec
 ARIMA(1,1,1)(0,0,0)[0] intercept   : AIC=13327.645, Time=0.25 sec
 ARIMA(2,1,1)(0,0,0)[0] intercept   : AIC=13323.026, Time=0.48 sec
 ARIMA(3,1,1)(0,0,0)[0] intercept   : AIC=13155.763, Time=0.47 sec
 ARIMA(3,1,0)(0,0,0)[0] intercept   : AIC=13189.282, Time=0.20 sec
 ARIMA(4,1,1)(0,0,0)[0] intercept   : AIC=13087.781, Time=0.59 sec
 ARIMA(4,1,0)(0,0,0)[0] intercept   : AIC=13115.377, Time=0.26 sec
 ARIMA(5,1,1)(0,0,0)[0] intercept   : AIC=13084.889, Time=0.67 sec
 ARIMA(5,1,0)(0,0,0)[0] intercept   : AIC=13083.152, Time=0.27 sec
 ARIMA(6,1,0)(0,0,0)[0] intercept   : AIC=13085.107, Time=0.33 sec
 ARIMA(6,1,1)(0,0,0

### Resultados de la selección automática ARIMA

El algoritmo auto_arima identificó como mejor configuración el modelo
ARIMA(5,1,0) según el criterio de información AIC.

Este resultado indica la presencia de dependencias autoregresivas de corto
alcance en la serie temporal, así como la necesidad de una diferenciación de
primer orden para estabilizar la media temporal. La ausencia de términos MA
(q = 0) sugiere que la dinámica del sistema puede capturarse adecuadamente
mediante componentes autoregresivos sin requerir modelado adicional del ruido.

La significancia estadística de los coeficientes y los tests de diagnóstico
confirman la adecuación del modelo en esta fase.


## Modelo ARIMA seleccionado

El algoritmo `auto_arima` identificó como mejor configuración el modelo
ARIMA(5,1,0) según el criterio de información AIC, utilizando una ventana
representativa del conjunto de entrenamiento.

A partir de este punto, el orden del modelo se mantiene fijo y se evalúa su
rendimiento mediante validación temporal walk-forward, comparando los resultados con los modelos baseline (Naive) para determinar si la modelización estadística aporta mejoras predictivas reales.


In [9]:
best_order = (5, 1, 0)
print("ARIMA seleccionado:", best_order)

ARIMA seleccionado: (5, 1, 0)


## Configuración de la evaluación

La evaluación se realiza utilizando el histórico acumulado disponible en cada
instante (train + validación), simulando un escenario real de predicción en el
que el modelo se reentrena periódicamente con todos los datos observados hasta
el momento.

Las métricas se calculan únicamente sobre el periodo de validación para evitar
fuga de información.

In [10]:
series_full = pd.concat([y_train, y_val]).sort_index()

val_start = y_val.index.min()
val_end   = y_val.index.max()

# Dimensión de la serie
print("Dimensión series_full:", series_full.shape)
print("Número de observaciones:", len(series_full))

# Rango temporal
print("Inicio:", series_full.index.min())
print("Fin:", series_full.index.max())

series_full.head()

Dimensión series_full: (105216,)
Número de observaciones: 105216
Inicio: 2022-01-01 00:00:00+00:00
Fin: 2024-12-31 23:45:00+00:00


timestamp
2022-01-01 00:00:00+00:00    120.79
2022-01-01 00:15:00+00:00    120.79
2022-01-01 00:30:00+00:00    120.79
2022-01-01 00:45:00+00:00    120.80
2022-01-01 01:00:00+00:00    120.04
Name: y, dtype: float64

In [11]:
from statsmodels.tsa.arima.model import ARIMA

def forecast_arima(train, h, order):
    
    model = ARIMA(
        train.values,
        order=order,
        trend="n",                   # sin intercepto 
        enforce_stationarity=False,
        enforce_invertibility=False
    )
    
    res = model.fit(method_kwargs={"warn_convergence": False})
    
    return res.forecast(steps=h)

In [12]:
import numpy as np
# Modelo base
def forecast_naive_last(train, h):
    """
    Predicción naive por persistencia:
    el futuro es igual al último valor observado.
    """
    last_val = train.values[-1]
    return np.repeat(last_val, h)



def forecast_naive_seasonal(train, h, season=96):
    """
    Predicción naive estacional:
    repite el patrón observado hace 24h.
    """
    
    if len(train) < season:
        return forecast_naive_last(train, h)
    
    pattern = train.values[-season:]
    
    # repetir patrón hasta cubrir horizonte
    reps = int(np.ceil(h / season))
    forecast = np.tile(pattern, reps)[:h]
    
    return forecast



## Evaluación walk-forward

En cada punto temporal:

1. El modelo se entrena con todo el histórico disponible.
2. Se generan predicciones para horizontes de 48h y 72h.
3. Se calculan métricas sobre el periodo de validación.

Este procedimiento proporciona una evaluación robusta y realista del modelo.

In [13]:
def walk_forward_splits(series, horizons, cfg):
    """
    Genera splits walk-forward sobre una serie temporal.
    
    - series: pd.Series con índice datetime
    - horizons: dict {"48h": steps, "72h": steps}
    - cfg: WFConfig(step, min_history, max_fits)
    
    Devuelve iteraciones:
      train_part: serie hasta t0 (excluye futuro)
      tests: dict con segmentos futuros para cada horizonte
    """
    n = len(series)
    max_h = max(horizons.values())

    # puntos t0 donde empezamos a predecir
    points = list(range(cfg.min_history, n - max_h, cfg.step))
    points = points[:cfg.max_fits]

    for t0 in points:
        train_part = series.iloc[:t0]
        tests = {k: series.iloc[t0:t0 + h] for k, h in horizons.items()}
        yield train_part, tests

In [14]:
def evaluate_model_on_validation(y_train,
                                 y_val,
                                 horizons,
                                 cfg,
                                 forecaster):

    rows = []

    for train_part, tests in walk_forward_splits(y_val, horizons, cfg):

        # Histórico real disponible
        train_full = pd.concat([y_train, train_part])

        for name, test in tests.items():

            pred = forecaster(train_full, len(test))

            y_true = test.values
            y_pred = pred

            m = compute_metrics(y_true, y_pred)

            rows.append({
                "horizon": name,
                "MAE": m["MAE"],
                "RMSE": m["RMSE"]
            })

    df = pd.DataFrame(rows)

    print("Filas generadas:", len(df))

    return df

In [15]:
naive_last_val = evaluate_model_on_validation(
    y_train,
    y_val,
    HORIZONS,
    cfg_final,
    forecast_naive_last
)

naive_seas_val = evaluate_model_on_validation(
    y_train,
    y_val,
    HORIZONS,
    cfg_final,
    lambda tr, h: forecast_naive_seasonal(tr, h)
)
naive_last_val.head()

Filas generadas: 240
Filas generadas: 240


,horizon,MAE,RMSE
0,48h,13.052135,16.205747
1,72h,23.059028,28.490951
2,48h,30.395885,34.404873
3,72h,31.111632,34.604865
4,48h,14.448490,24.198182


In [16]:
def summarize(df, model_name):
    """
    Resume métricas medias por horizonte para un modelo.
    """
    
    out = (
        df.groupby("horizon")[["MAE", "RMSE"]]
        .mean()
        .reset_index()
    )
    
    out.insert(0, "model", model_name)
    
    return out

In [17]:
naive_summary = pd.concat([
    summarize(naive_last_val, "Naive_last"),
    summarize(naive_seas_val, "Naive_seasonal")
])

naive_summary

,model,horizon,MAE,RMSE
0,Naive_last,48h,20.174324,24.491983
1,Naive_last,72h,21.887282,26.416587
0,Naive_seasonal,48h,16.510456,20.179678
1,Naive_seasonal,72h,17.744089,21.888852


El modelo Naive estacional presenta un mejor rendimiento que la persistencia
simple, lo cual es consistente con la fuerte periodicidad diaria presente en
las series eléctricas. Por este motivo, el naive estacional se considera el
baseline principal para la comparación con modelos más complejos.

In [18]:
arima_val = evaluate_model_on_validation(
    y_train,
    y_val,
    HORIZONS,
    cfg_final,
    lambda tr, h: forecast_arima(tr, h, best_order)
)

arima_summary = summarize(arima_val, f"ARIMA{best_order}")

Filas generadas: 240


In [19]:
compare_val = pd.concat([
    naive_summary,
    arima_summary
], ignore_index=True)

compare_val

,model,horizon,MAE,RMSE
0,Naive_last,48h,20.174324,24.491983
1,Naive_last,72h,21.887282,26.416587
2,Naive_seasonal,48h,16.510456,20.179678
3,Naive_seasonal,72h,17.744089,21.888852
4,"ARIMA(5, 1, 0)",48h,20.414305,24.609350
5,"ARIMA(5, 1, 0)",72h,21.911410,26.260707


Los resultados muestran que el modelo ARIMA no estacional no supera al naive
estacional, lo cual indica que la dinámica principal de la serie está dominada
por patrones periódicos diarios. Este comportamiento es esperable en sistemas
eléctricos, donde la demanda y el mix de generación presentan ciclos regulares
de 24 horas.

Estos resultados justifican la necesidad de incorporar explícitamente componentes
estacionales en el modelo, lo que motiva la evaluación de modelos SARIMA en la
siguiente sección.



In [21]:
RESULTS_DIR = Path.cwd().parent / "results"
RESULTS_DIR.mkdir(exist_ok=True)

naive_summary.to_csv(RESULTS_DIR / "naive_summary.csv", index=False)
arima_summary.to_csv(RESULTS_DIR / "arima_summary.csv", index=False)

print("Guardado en:", RESULTS_DIR.resolve())

Guardado en: /home/ubuntu/TFM/results
